In [13]:
import openeo
from openeo.processes import quantiles

import xarray as xr
import os

In [2]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

## Sentinel-2 mosaic from Sentinel-2 L2A

Create the Sentinel-2 mosaic from L2A data. Product description: 

https://dataspace.copernicus.eu/news/2024-2-27-exploring-new-frontier-sentinel-cloudless-mosaics-copernicus-data-space-ecosystem

https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-3-quarterly-mosaics

In [14]:
spatial_extent = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
#spatial_extent = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers
temporal_extent = ["2017-07-01T00:00:00Z", "2017-09-30T23:59:59Z"]

output_folder = "/mnt/CEPH_PROJECTS/MASSIVE/Glacier_outline_ST/test/"

### 1) Generate the Sentinel-2 L2A cube

Load all the 10 m resolution bands + SCL

In [4]:
s2_cube_10m = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B02", "B03", "B04", "B08", "SCL"],
)

Add the B11 bilinearly resampled at 10 meters

In [5]:
s2_cube_B11 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B11"],
)

s2_cube_B11 = s2_cube_B11.resample_cube_spatial(target=s2_cube_10m, method="bilinear")
s2_cube = s2_cube_10m.merge_cubes(s2_cube_B11)

Add the band ratio B04/B11

In [6]:
s2_cube_bRatio = s2_cube.band("B04") / s2_cube.band("B11")
s2_cube_bRatio = s2_cube_bRatio.add_dimension("bands", "B04_B11_ratio", type="bands")
s2_cube = s2_cube_10m.merge_cubes(s2_cube_bRatio)

### 2) Create the mask from SCL

Create the mask using the following SCL values:
```
1  = SC_SATURATED_DEFECTIVE
3  = SC_CLOUD_SHADOW
7  = SC_CLOUD_LOW_PROBA / UNCLASSIFIED
8  = SC_CLOUD_MEDIUM_PROBA
9  = SC_CLOUD_HIGH_PROBA
10 = SC_THIN_CIRRUS
```

In [7]:
scl = s2_cube.band("SCL")
invalid_mask = (
    (scl == 1) |
    (scl == 3) |
    (scl == 7) |
    (scl == 8) |
    (scl == 9) |
    (scl == 10)
)

valid_mask = ~invalid_mask
# valid_mask.download('valid_mask.nc', format='NetCDF')

### 3) Generate the layer of valid observation

In [8]:
observations = valid_mask.reduce_dimension(dimension='t', reducer='sum')
observations = observations.add_dimension("bands", "observations", type="bands")
#observations.download('observations_2Jul_1Oct.nc', format='NetCDF')

### 4) Compute the first quartile of valid observations

In [9]:
s2_cube = s2_cube.filter_bands(["B02", "B03", "B04", "B08", "B04_B11_ratio"])

s2_mosaic = s2_cube.mask(invalid_mask).reduce_dimension(
    dimension='t',
    reducer=lambda x: quantiles(data=x, probabilities=[0.25])
)
#s2_mosaic.download('s2_mosaic.nc', format='NetCDF')

### 5) Export the result

In [10]:
s2_mosaic = s2_mosaic.merge_cubes(observations)
#s2_mosaic.download('/mnt/CEPH_PROJECTS/MASSIVE/Glacier_outline_ST/s2_mosaic_2023_ST_west.nc', format='NetCDF')
job = s2_mosaic.create_job(title='S2_mosaic')
job.start()

<BatchJob job_id='j-2606030640584ef7af3ec173b96e6d20'>

In [11]:
job

<BatchJob job_id='j-2606030640584ef7af3ec173b96e6d20'>

In [12]:
job.get_results().download_files(output_folder)

[PosixPath('/mnt/CEPH_PROJECTS/MASSIVE/Glacier_outline_ST/test/openEO.tif'),
 PosixPath('/mnt/CEPH_PROJECTS/MASSIVE/Glacier_outline_ST/test/job-results.json')]

In [16]:
os.path.join(output_folder, "openEO.tif")

'/mnt/CEPH_PROJECTS/MASSIVE/Glacier_outline_ST/test/openEO.tif'

In [17]:
#xr.open_dataset(os.path.join(output_folder, "openEO.nc"))

## Sentinel-2 mosaic with Sentinel-hub evalscript

```javascript
//VERSION=3
/*
Script works on Sentinel-2 L2A data and requires scene classification (SCL) band. 
It takes one year of data, which is quite compute and time intensive, which is why it is recommended to run it on small area (e.g. 256x256 px).
An example of the results is New Zealand's cloudless mosaic, available here: https://data.linz.govt.nz/layer/93652-nz-10m-satellite-imagery-2017/
For the output value for each pixel it uses the first quartile value of valid values, each band separately. If there are none it uses invalid values instead. 
When using SCL its very important to use nearest neighbor resampling with a resolution of about 20m/px or more. 
*/

function setup() {
  return {
    input: [{
      bands: [
        "B04",
        "B03",
        "B02",
        "SCL"
      ]
    }],
    output: { bands: 3, sampleType: "UINT16" },
    mosaicking: "ORBIT"
  }
}
function preProcessScenes(collections) {
  collections.scenes.orbits = collections.scenes.orbits.filter(function (orbit) {
    var orbitDateFrom = new Date(orbit.dateFrom)
    return orbitDateFrom.getTime() >= (collections.to.getTime() - 3 * 31 * 24 * 3600 * 1000);
  })
  return collections
}
function getValue(values) {
  values.sort(function (a, b) { return a - b; });
  return getFirstQuartile(values);
}

function getFirstQuartile(sortedValues) {
  var index = Math.floor(sortedValues.length / 4);
  return sortedValues[index];
}
function getDarkestPixel(sortedValues) {
  return sortedValues[0]; // darkest pixel
}
function validate(samples) {
  var scl = samples.SCL;

  if (scl === 3) { // SC_CLOUD_SHADOW
    return false;
  } else if (scl === 9) { // SC_CLOUD_HIGH_PROBA
    return false;
  } else if (scl === 8) { // SC_CLOUD_MEDIUM_PROBA
    return false;
  } else if (scl === 7) { // SC_CLOUD_LOW_PROBA / UNCLASSIFIED
    // return false;
  } else if (scl === 10) { // SC_THIN_CIRRUS
    return false;
  } else if (scl === 11) { // SC_SNOW_ICE
    return false;
  } else if (scl === 1) { // SC_SATURATED_DEFECTIVE
    return false;
  } else if (scl === 2) { // SC_DARK_FEATURE_SHADOW
    // return false;
  }
  return true;
}

function evaluatePixel(samples, scenes) {
  var clo_b02 = []; var clo_b03 = []; var clo_b04 = [];
  var clo_b02_invalid = []; var clo_b03_invalid = []; var clo_b04_invalid = [];
  var a = 0; var a_invalid = 0;

  for (var i = 0; i < samples.length; i++) {
    var sample = samples[i];

    if (sample.B02 > 0 && sample.B03 > 0 && sample.B04 > 0) {
      var isValid = validate(sample);

      if (isValid) {
        clo_b02[a] = sample.B02;
        clo_b03[a] = sample.B03;
        clo_b04[a] = sample.B04;
        a = a + 1;
      } else {
        clo_b02_invalid[a_invalid] = sample.B02;
        clo_b03_invalid[a_invalid] = sample.B03;
        clo_b04_invalid[a_invalid] = sample.B04;
        a_invalid = a_invalid + 1;
      }
    }
  }

  var rValue;
  var gValue;
  var bValue;
  if (a > 0) {
    rValue = getValue(clo_b04);
    gValue = getValue(clo_b03);
    bValue = getValue(clo_b02);
  } else if (a_invalid > 0) {
    rValue = getValue(clo_b04_invalid);
    gValue = getValue(clo_b03_invalid);
    bValue = getValue(clo_b02_invalid);
  } else {
    rValue = 0;
    gValue = 0;
    bValue = 0;
  }
  return [rValue * 10000,
  gValue * 10000,
  bValue * 10000]
}
```

## Download Sentinel-2 Copernicus mosaic and Sentinel-2 L2A for comparison

Download the Sentinel-2 copernicus mosaic https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-global-mosaics

In [11]:
s2_mosaic_cube = connection.load_stac(
    url="https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-global-mosaics",
    spatial_extent=spatial_extent,
    temporal_extent=["2025-07-01T00:00:00Z", "2025-07-01T23:59:59Z"],
)

s2_mosaic_cube = s2_mosaic_cube.resample_spatial()

s2_mosaic_cube.download('s2_mosaic_01Jul2025.nc', format='NetCDF')

Download Sentinel-2A

In [10]:
s2_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=["2025-08-17", "2025-08-19"],
    bands=["B02", "B03", "B04", "SCL"],
)
s2_cube.download('s2_18Aug2025.nc', format='NetCDF')

In [11]:
xr.open_dataset("s2_18Aug2025.nc")

<xarray.Dataset> Size: 31MB
Dimensions:  (t: 1, x: 1561, y: 1255)
Coordinates:
  * t        (t) datetime64[ns] 8B 2025-08-18
  * x        (x) float64 12kB 6.188e+05 6.188e+05 ... 6.344e+05 6.344e+05
  * y        (y) float64 10kB 5.157e+06 5.157e+06 ... 5.144e+06 5.144e+06
Data variables:
    crs      |S1 1B ...
    B02      (t, y, x) float32 8MB ...
    B03      (t, y, x) float32 8MB ...
    B04      (t, y, x) float32 8MB ...
    SCL      (t, y, x) float32 8MB ...
Attributes:
    Conventions:  CF-1.9
    institution:  openEO platform